In [ ]:
from src.utils.feature_utils import load_data
from src.logging.logger import ExperimentLogger
from src.utils.model_utils import CatBoostWrapper
from src.utils.model_utils import run_universal_cv
import pandas as pd

We are using our custom logger for the experiment and load parquet files for optimization purposes

In [ ]:
logger = ExperimentLogger()

X_train, X_test = load_data(path_to_data_folder='../data/processed', file_type='parquet')

In [ ]:
X_train.head()

In [ ]:
TARGET = 'loan_paid_back'
y = X_train[TARGET]
X_train = X_train.drop(columns=TARGET)

We have to provide a list of all categorical columns for our CatBoost model

In [ ]:
cats = X_train.select_dtypes(include=['string', 'category', 'object']).columns.tolist()

In [ ]:
cats

We use parameters obtained from trials using Optuna library to build the model

In [ ]:
model_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 25000,
    'learning_rate': 0.01309055676624934,
    'depth': 5,
    'l2_leaf_reg': 5.479345908756687,
    "random_strength": 1.0120533017977982,
    "bagging_temperature": 0.7674209193844299,
    'random_seed': 42,
    'task_type': 'GPU',
    'early_stopping_rounds': 500,
    'verbose': 500
}

model = CatBoostWrapper(model_params, cat_features=cats)

Now we run a cross validation loop using our model and processed data

In [ ]:
oof_predictions, test_predictions, cv_auc, mean_importances_df = run_universal_cv(X_train=X_train, y_train=y, X_test=X_test, model_wrapper=model, preprocessor_func=None)

We save the results of the experiment and test set predictions in a dedicated logs folder

In [ ]:
logger.log_experiment(exp_name="CatBoost_Experiment", model_name="CatBoost", cv_score=cv_auc, params=model_params, features_list=X_train.columns.tolist(), importances_df=mean_importances_df)

In [ ]:
oof_df = pd.DataFrame({
    'id': X_train.index,
    'loan_paid_back': oof_predictions
})
oof_df.to_csv('../logs/oof_CatBoost.csv', index=False)

test_df_out = pd.DataFrame({
    'id': X_test.index,
    'loan_paid_back': test_predictions
})
test_df_out.to_csv('../logs/CatBoost_predictions.csv', index=False)

# Conclusions
## 1. Feature engineering and domain knowledge
Instead of relying solely on raw data, we implemented a series of transformations based on banking sector knowledge and the specifics of synthetic data:

- Risk indicators and Scorecard: We constructed synthetic variables assessing the client's profile, such as default_risk, expected_loss, and risk_adjusted_return. Additionally, we implemented a custom_scorecard, which scored credit applications based on the 5C framework. Although not every method worked great, there were some which highly contributed to the signal.

- Target encoding and pseudo-TE:  Categorical variables were encoded using historical averages from the original dataset. Thanks to this, the model received a direct signal about the probability of repayment for a given category, avoiding data leakage in the validation loop.

- Data leakage exploitation (KNN): Using the K-Nearest Neighbors algorithm, we "borrowed" key variables unavailable in the synthetic dataset (e.g. age, current_balance). This filled significant information gaps and allowed for a more precise assessment of creditworthiness.

- Noise reduction (binning & round hack): To mitigate artifacts introduced by the synthetic data generator, numerical variables (e.g. loan_amount) were discretized and rounded. This made it easier for decision trees to find stable split points.

## 2. Feature Evaluation and Selection
- Not every generated variable added value to the model. We based the evaluation of their effectiveness on the analysis of feature importances.

- After generating a broad feature space (over 90 variables), we analyzed their contribution to error reduction.

- It turned out that the newly created domain features such as default_risk and TE statistics such as TE_orig_employment_status were among the top predictors.

- We applied a hard cutoff threshold (weight < 0.1), discarding the weakest features. This allowed us to clean the dataset of noise.